# Import

In [18]:
import os
from google.colab import drive
import numpy as np
import scipy.io
from scipy.signal import convolve2d
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import numpy as np

drive.mount('/content/drive')
path_to_project = '/content/drive/MyDrive/IC/projetIC/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Tranformations des .mat en .png

In [ ]:
def mat_to_png(mat_path, is_rf=False):
    mat = scipy.io.loadmat(mat_path)
    best_data = None
    max_size = 0
    best_key = ""

    for key, val in mat.items():
        if key.startswith('__'): continue
        if isinstance(val, np.ndarray) and np.issubdtype(val.dtype, np.number):
            if val.ndim >= 2:
                if val.size > max_size:
                    max_size = val.size
                    best_data = val
                    best_key = key

    if best_data is None:
        print("Aucune image trouvée dans ce fichier (pas de matrice numérique).")
        return
    img_data = np.squeeze(best_data)
    if is_rf:
        img_data = np.abs(img_data)
    img_data = img_data.astype(float)
    min_val, max_val = img_data.min(), img_data.max()
    if max_val - min_val == 0:
        print("L'image est uniforme (contraste nul).")
        data_norm = np.zeros_like(img_data)
    else:
        data_norm = (img_data - min_val) / (max_val - min_val) * 255
    output_filename = os.path.splitext(mat_path)[0] + ".png"
    final_image = Image.fromarray(data_norm.astype('uint8'))
    final_image.save(output_filename)
    print(f"Image sauvegardée : {output_filename}, de taille {data_norm.shape}")

mat_to_png('/content/drive/MyDrive/IC/projetIC/simu/1/psf_GT.mat')

Image sauvegardée : /content/drive/MyDrive/IC/projetIC/simu/1/psf_GT.png, de taille (11, 9)


## 4 images simu

In [ ]:
set_name = "simu/"
filename = "psf_GT.mat"
for i in range(1,5):
  path = "{}{}{}/{}".format(path_to_project, set_name, i, filename)
  mat_to_png(path)

Image sauvegardée : /content/drive/MyDrive/IC/projetIC/simu/1/psf_GT.png, de taille (11, 9)
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/simu/2/psf_GT.png, de taille (11, 9)
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/simu/3/psf_GT.png, de taille (11, 9)
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/simu/4/psf_GT.png, de taille (11, 9)


## images vivo

In [ ]:
set_name = "vivo/"
dir_name = ["invivo2/", "carotid/", "carotid/"]
filename = ["psf_est.mat", "psf.mat", "bmode.mat"]
for i in range(len(dir_name)):
  path = "{}{}{}{}".format(path_to_project, set_name, dir_name[i], filename[i])
  mat_to_png(path)

Image sauvegardée : /content/drive/MyDrive/IC/projetIC/vivo/invivo2/psf_est.png, de taille (31, 31)
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/vivo/carotid/psf.png, de taille (4480, 4480)
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/vivo/carotid/bmode.png, de taille (4480, 384)


# Get low-resolution

In [ ]:
def get_lr(x_png, h_png, S=4, sigma=0.001):
    x = np.array(Image.open(x_png).convert("L"), dtype=np.float64)
    x = x / 255.0

    h = np.array(Image.open(h_png).convert("L"), dtype=np.float64)
    h = h / np.sum(h)

    hXx = convolve2d(x, h, mode="same", boundary="symm")

    y = hXx[::S, ::S]

    y = cv2.resize(y, (64, 64), interpolation=cv2.INTER_AREA)

    n = sigma * np.random.randn(*y.shape)
    return y + n


## 4 images simu

In [ ]:
set_name = "simu/"
set_name_out = "projet_robin/inputs/simu/"
filename_psf = "psf_GT.png"
filename_gt = "bmode_GT.png"
for i in range(1,5):
  path_psf = "{}{}{}/{}".format(path_to_project, set_name, i, filename_psf)
  path_gt = "{}{}{}/{}".format(path_to_project, set_name, i, filename_gt)
  output_filename = "{}{}{}{}{}".format(path_to_project, set_name_out, "bmode_LR", i, ".png")
  im_lr = get_lr(path_gt, path_psf)
  im_lr = np.clip(im_lr, 0, 1) * 255
  plt.imsave(output_filename, im_lr, cmap="gray")
  print(f"Image sauvegardée : {output_filename} Min: {im_lr.min()}, Max: {im_lr.max()}, Mean: {im_lr.mean()}")

Image sauvegardée : /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR1.png Min: 14.62673128450809, Max: 111.08658450730573, Mean: 56.75578193751713
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR2.png Min: 14.385077899101809, Max: 108.39292018430142, Mean: 51.91976599408214
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR3.png Min: 19.42069825090417, Max: 100.20119443983826, Mean: 72.91647627412581
Image sauvegardée : /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR4.png Min: 19.244204580559707, Max: 100.15576863154256, Mean: 72.90737117618033


## images vivo

In [ ]:
set_name_out = "projet_robin/inputs/vivo/"
set_name = "vivo/"
dir_name = ["invivo2/"]
filename_psf = ["psf_est.png"]
filename_gt = "bmode.png"
for i in range(len(dir_name)):
  path_psf = "{}{}{}{}".format(path_to_project, set_name, dir_name[i], filename_psf[i])
  path_gt = "{}{}{}{}".format(path_to_project, set_name, dir_name[i], filename_gt)
  output_filename = "{}{}{}{}{}".format(path_to_project, set_name_out, "bmode_LR", i, ".png")
  im_lr = get_lr(path_gt, path_psf)
  im_lr = np.clip(im_lr, 0, 1) * 255
  plt.imsave(output_filename, im_lr, cmap="gray")
  print(f"Image sauvegardée : {output_filename} Min: {im_lr.min()}, Max: {im_lr.max()}, Mean: {im_lr.mean()}")

Image sauvegardée : /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/vivo/bmode_LR0.png Min: 2.190083037861604, Max: 255.0, Mean: 69.18050550111603


In [24]:
import torch
import PIL
import torchvision
transform = torchvision.transforms.Compose([
		torchvision.transforms.Resize(512),
		torchvision.transforms.CenterCrop(512),
	])

def load_img(path):
	image = Image.open(path).convert("RGB")
	w, h = image.size
	print(f"loaded input image of size ({w}, {h}) from {path}")
	w, h = map(lambda x: x - x % 32, (w, h))  # resize to integer multiple of 32
	image = image.resize((w, h), resample=PIL.Image.LANCZOS)
	image = np.array(image).astype(np.float32) / 255.0
	image = image[None].transpose(0, 3, 1, 2)
	image = torch.from_numpy(image)
	print_stats("AVANT 2.*x-1 ", image)
	return 2.*image - 1.

def print_stats(name, tensor):
    """Affiche les statistiques min, max et moyenne d'un tenseur."""
    t_min = tensor.min().item()
    t_max = tensor.max().item()
    t_mean = tensor.mean().item()
    print(f"[{name}] Min: {t_min:.4f} | Max: {t_max:.4f} | Mean: {t_mean:.4f}")
    return

img = load_img('/content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR1.png')
print_stats("APRES ", img)
img = transform(img)
print_stats("APRES ", img)

loaded input image of size (64, 64) from /content/drive/MyDrive/IC/projetIC/projet_robin/inputs/simu/bmode_LR1.png
[AVANT 2.*x-1 ] Min: 0.0000 | Max: 1.0000 | Mean: 0.4362
[APRES ] Min: -1.0000 | Max: 1.0000 | Mean: -0.1276
[APRES ] Min: -0.9984 | Max: 0.9956 | Mean: -0.1276
